![](img/logo_ucm.jpg)

# Ejercicio práctico-teórico resuelto: MLflow 

En este ejercicio vas a consolidar los conceptos principales del tema 3: **tracking de experimentos**, **comparacion de runs**, **artefactos** y **carga de modelos desde MLflow**. 

Trabajaremos con el dataset `diabetes` de `scikit-learn` para evitar dependencias externas y poder centrarnos en el flujo de MLflow. El objetivo del ejercicio es afianzar conocimientos entrenando modelos y dejando evidencia reproducible de su flujo de construcción y evaluación. 

## Objetivos

1. Configurar un tracking local con MLflow.
2. Crear un experimento y registrar varios modelos como runs anidados.
3. Loggear parametros, metricas, tags, artefactos y el modelo entrenado.
4. Comparar los runs y seleccionar el mejor segun RMSE.
5. Cargar el mejor modelo desde una URI `runs:/...`.
6. Razonar las diferencias entre metricas, artefactos y trazabilidad.

In [8]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Apartado 1: configuracion del experimento

Configura un tracking local para este ejercicio usando SQLite como backend de metadata, crea las carpetas necesarias y activa el experimento `tema3-mlflow`.

In [9]:
EXPERIMENT_NAME = "tema3-mlflow"
TRACKING_DB = Path("mlflow_tarea_mlflow.db")
ARTIFACTS_DIR = Path("artifacts_tarea_mlflow")

ARTIFACTS_DIR.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


2026/06/07 19:26:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 19:26:58 INFO mlflow.store.db.utils: Updating database tables
2026/06/07 19:26:59 INFO mlflow.tracking.fluent: Experiment with name 'tema3-mlflow' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow_tarea_mlflow.db
Experiment: tema3-mlflow


### Revisar los experimentos en la interfaz de MLflow

Una vez ejecutado el notebook, podemos levantar la interfaz web de MLflow para inspeccionar los runs, comparar metricas, revisar parametros y navegar por los artifacts registrados.

```bash
cd notebooks_clase/tema_3_mlflow
uv run mlflow ui --backend-store-uri sqlite:///mlflow_tarea_mlflow.db --port 5000
```

Despues, abre `http://127.0.0.1:5000` en el navegador.


## Apartado 2: carga y particion del dataset

Carga `diabetes`, separa `X` e `y` y haz una particion train/test con `test_size=0.2` y `random_state=42`.

In [10]:
dataset = load_diabetes(as_frame=True)
df = dataset.frame.copy()

FEATURE_NAMES = dataset.feature_names
TARGET_NAME = "target"

X = df[FEATURE_NAMES]
y = df[TARGET_NAME]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(df.head(3))
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
Train shape: (353, 10)
Test shape: (89, 10)


## Apartado 3: helpers de evaluación

Usaremos tres candidatos sencillos de regresion y una funcion auxiliar para calcular métricas y generar un gráfico de residuos.

In [11]:
def build_candidates():
    return {
        "linear_regression": Pipeline(
            steps=[("scaler", StandardScaler()), ("model", LinearRegression())]
        ),
        "random_forest": Pipeline(
            steps=[(
                "model",
                RandomForestRegressor(
                    n_estimators=200,
                    max_depth=6,
                    random_state=42,
                    n_jobs=-1,
                ),
            )]
        ),
        "gradient_boosting": Pipeline(
            steps=[("model", GradientBoostingRegressor(random_state=42))]
        ),
    }


def regression_metrics(y_true, y_pred):
    return {
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def make_residual_plot(y_true, y_pred, title, output_path):
    residuals = y_true - y_pred
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(y_pred, residuals, alpha=0.7)
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Residuals - {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual")
    fig.tight_layout()
    fig.savefig(output_path, dpi=120)
    plt.close(fig)


## Apartado 4: entrenamiento y tracking con nested runs

Para cada candidato, entrena el pipeline, calcula métricas y registra en MLflow:

- parametros del estimador final
- metricas de test (`rmse`, `mae`, `r2`)
- tags útiles
- modelo en `artifact_path="model"`
- gráfico de residuos como artifact

In [12]:
run_summaries = []

with mlflow.start_run(run_name="model-comparison-diabetes") as parent_run:
    mlflow.set_tag("exercise", "tema_3_mlflow")
    mlflow.set_tag("dataset", "diabetes")

    for candidate_name, pipeline in build_candidates().items():
        with mlflow.start_run(run_name=f"candidate-{candidate_name}", nested=True) as child_run:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            metrics = regression_metrics(y_test, y_pred)

            final_model = pipeline.named_steps["model"]
            model_params = final_model.get_params()

            mlflow.log_params(model_params)
            mlflow.log_metrics(metrics)
            mlflow.set_tags(
                {
                    "candidate_name": candidate_name,
                    "problem_type": "regression",
                    "target_name": TARGET_NAME,
                }
            )

            signature = infer_signature(X_train, pipeline.predict(X_train))
            mlflow.sklearn.log_model(
                sk_model=pipeline,
                artifact_path="model",
                signature=signature,
                input_example=X_train.head(3),
            )

            plot_path = ARTIFACTS_DIR / f"residuals_{candidate_name}.png"
            make_residual_plot(y_test, y_pred, candidate_name, plot_path)
            mlflow.log_artifact(plot_path)

            run_summaries.append(
                {
                    "candidate_name": candidate_name,
                    "run_id": child_run.info.run_id,
                    **metrics,
                }
            )

leaderboard = pd.DataFrame(run_summaries).sort_values("rmse").reset_index(drop=True)
leaderboard


2026/06/07 19:27:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 19:27:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/07 19:27:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 19:27:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

,candidate_name,run_id,rmse,mae,r2
0,random_forest,48c1f1b8739447e98a6f63bef64cf957,53.829500,43.638079,0.453089
1,gradient_boosting,2f43073302e44730b2c8bd8de27e65b5,53.837131,44.603297,0.452934
2,linear_regression,ad8569f77ea644c19ce176523a205957,53.853446,42.794095,0.452603


## Apartado 5: selección del mejor run y carga del modelo

Selecciona el mejor run segun menor RMSE, construye una URI `runs:/.../model` y carga el modelo con `mlflow.pyfunc.load_model`.

In [13]:
best_row = leaderboard.iloc[0]
best_model_uri = f"runs:/{best_row['run_id']}/model"
best_model = mlflow.pyfunc.load_model(best_model_uri)

sample_predictions = best_model.predict(X_test.head(3))

print("Best run:")
print(best_row)
print("Model URI:", best_model_uri)
print("Sample predictions:", sample_predictions)


Best run:
candidate_name                       random_forest
run_id            48c1f1b8739447e98a6f63bef64cf957
rmse                                       53.8295
mae                                      43.638079
r2                                        0.453089
Name: 0, dtype: object
Model URI: runs:/48c1f1b8739447e98a6f63bef64cf957/model
Sample predictions: [152.93539791 173.28943658 151.6491704 ]


## Apartado 6: inspeccion con MlflowClient

Recupera el experimento y el mejor run usando `MlflowClient`. Despues lista los artifacts registrados en la raiz del run.

In [14]:
client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment ID:", experiment.experiment_id if experiment else "not found")

best_run = client.get_run(best_row["run_id"])
artifacts = client.list_artifacts(best_row["run_id"])

print("Best run params:", best_run.data.params)
print("Best run metrics:", best_run.data.metrics)
print("Artifacts:", [artifact.path for artifact in artifacts])


Experiment ID: 1
Best run params: {'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': '6', 'max_features': '1.0', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '2', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '200', 'n_jobs': '-1', 'oob_score': 'False', 'random_state': '42', 'verbose': '0', 'warm_start': 'False'}
Best run metrics: {'rmse': 53.82950035526519, 'mae': 43.63807933059561, 'r2': 0.4530894458980377}
Artifacts: ['residuals_random_forest.png']


## Preguntas teóricas resueltas

1. ¿Qué ventaja aporta usar nested runs frente a registrar todos los modelos candidatos en un único run? Los nested runs permiten encapsular cada candidato bajo un contexto común facilitando comparar varias alternativas sin perder la relación con una ejecución padre de selección de modelo.
2. ¿En qué se diferencian parámetros, métricas, tags y artifacts dentro de MLflow? Los parámetros describen la configuración del modelo, las métricas cuantifican su rendimiento, los tags añaden contexto semántico y los artifacts guardan ficheros producidos durante el experimento.
3. ¿Por qué puede ser más razonable seleccionar el mejor modelo por `rmse` en lugar de por `r2` en este problema? `rmse` está en las mismas unidades que la variable objetivo y penaliza más los errores grandes, por lo que suele ser muy interpretable cuando interesa controlar desviaciones relevantes de la predicción.
4. ¿Qué información adicional necesitarias registrar si quisieras auditar completamente un experimento meses después? Conviene registrar version del código, dataset o hash del dataset, semilla, signature del modelo, ejemplos de entrada, dependencias y criterios exactos de selección del ganador.
5. ¿Qué limitaciones prácticas tiene usar un tracking local basado en ficheros frente a un tracking server compartido? Un tracking local en ficheros dificulta el trabajo colaborativo, el acceso concurrente, la persistencia y la portabilidad. Un tracking server compartido centraliza el metadato y mejora la operativa de equipo.

## Extensiones propuestas

1. Registrar un `leaderboard.csv` como artifact del run padre.
2. Añadir una cuarta familia de modelo y justificar si mejora o no al resto.
3. Registrar un alias o nombre estable del mejor modelo usando Model Registry en un servidor MLflow con backend SQL.
